In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from sometria.catalog import MotionViewSpec, build_motion_view
from sometria.preprocess import (
    ImportConfig,
    _load_human_definition,
    import_opensim_csv_dataset,
    import_babel_splits,
    create_pretrain_split,
    save_feature_normalization,
    feature_dofs,
    feature_normalization_mask,
)


In [2]:
RAW_AMASS = Path("/home/z1ko/datasets/sometria/amass")
RAW_BABEL = Path("/home/z1ko/datasets/sometria/babel")
OUT = Path("../data/processed")

REPRESENTATION = "opensim_sincos_log_vel_acc_tau_v2"
NORMALIZATION_NAME = "pretrain_v1_train_clean"
NORMALIZATION_PATH = OUT / "stats" / REPRESENTATION / f"{NORMALIZATION_NAME}.pt"


In [ ]:
human_def = _load_human_definition("/home/z1ko/develop/sometria/config/human.yaml")
catalog = import_opensim_csv_dataset(
    config=ImportConfig(
        source_dataset="AMASS",
        representation=REPRESENTATION,
        input_root=RAW_AMASS,
        output_root=OUT,
        pattern="**/*.csv"
    ),
    human=human_def
)

babel_splits = import_babel_splits(
    babel_root=RAW_BABEL,
    output_root=OUT
)

pretrain = create_pretrain_split(output_root=OUT)

catalog.head()


 24%|██▎       | 4134/17453 [02:25<07:45, 28.62it/s]

## Normalization


In [ ]:
train_samples = build_motion_view(
    OUT,
    MotionViewSpec(
        split_set="pretrain_v1",
        split="train",
        source_datasets=("AMASS",),
        exclude_broken=True,
    )
)

human_def = _load_human_definition("/home/z1ko/develop/sometria/config/human.yaml")
normalization_path = save_feature_normalization(
    output_root=OUT,
    samples=train_samples,
    name=NORMALIZATION_NAME,
    representation=REPRESENTATION,
    split_set="pretrain_v1",
    split="train",
    normalization_mask=feature_normalization_mask(human_def),
)

normalization_path


NameError: name 'human_def' is not defined

## Catalog sanity checks

These cells inspect the final AMASS motion catalog and BABEL-derived split tables. They are meant to answer: what was imported, what is labelled, what is clean, and what will each training view actually see?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch as t

from sometria.catalog import CATALOG, SPLITS

TABLES = OUT / "tables"
catalog = pl.read_parquet(TABLES / CATALOG)
splits = pl.read_parquet(TABLES / SPLITS)

babel_ids = (
    splits
    .filter(pl.col("split_set") == "babel_official")
    .select("sample_id")
    .unique()
    .get_column("sample_id")
    .to_list()
)

catalog = catalog.with_columns(
    pl.col("sample_id").is_in(babel_ids).alias("has_babel_split")
)

print(f"{len(catalog):,} motion samples")
print(f"{catalog['duration'].sum() / 3600:.1f} motion hours")
print(f"{catalog['source_subset'].n_unique()} AMASS subsets")
if {"n_dofs", "n_features"} <= set(catalog.columns):
    print(f"feature shape: {catalog['n_dofs'][0]} dofs x {catalog['n_features'][0]} features")
else:
    print("feature shape: rerun the import cell to populate n_dofs/n_features")
print(f"{catalog['has_babel_split'].sum():,} samples matched to BABEL official splits")
if NORMALIZATION_PATH.exists():
    stats = t.load(NORMALIZATION_PATH, weights_only=True)
    print(
        f"normalization: {NORMALIZATION_PATH} "
        f"({stats['n_samples']:,} samples, {stats['n_frames']:,} frames)"
    )
else:
    print(f"normalization: missing ({NORMALIZATION_PATH})")

catalog.head(3)


### Coverage overview

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

clean = (~catalog["broken"]).to_numpy()
has_babel = catalog["has_babel_split"].to_numpy()

ax[0].bar(
    ["all", "clean", "BABEL split"],
    [len(catalog), int(clean.sum()), int(has_babel.sum())],
    color=["#777777", "#4878a8", "#6acc64"],
)
ax[0].set(title="Sample counts", ylabel="samples")

hours = [
    catalog["duration"].sum() / 3600,
    catalog.filter(~pl.col("broken"))["duration"].sum() / 3600,
    catalog.filter(pl.col("has_babel_split"))["duration"].sum() / 3600,
]
ax[1].bar(["all", "clean", "BABEL split"], hours, color=["#777777", "#4878a8", "#6acc64"])
ax[1].set(title="Motion hours", ylabel="hours")

split_order = {"train": 0, "val": 1, "test": 2}
split_counts = (
    splits
    .filter(pl.col("split_set") == "babel_official")
    .group_by("split")
    .len()
    .with_columns(pl.col("split").replace(split_order).cast(pl.Int64).alias("order"))
    .sort("order")
)
ax[2].bar(split_counts["split"], split_counts["len"], color="#d1615d")
ax[2].set(title="BABEL official split", ylabel="samples")

fig.tight_layout()

### Per-dataset summary

In [ ]:
dataset_summary = (
    catalog
    .group_by("source_subset")
    .agg(
        pl.len().alias("samples"),
        (~pl.col("broken")).sum().alias("clean_samples"),
        pl.col("has_babel_split").sum().alias("babel_samples"),
        (pl.col("duration").sum() / 3600).alias("motion_hours"),
        (pl.when(~pl.col("broken")).then(pl.col("duration")).otherwise(0).sum() / 3600).alias("clean_motion_hours"),
        pl.col("broken").sum().alias("broken_samples"),
    )
    .with_columns(
        (100 * pl.col("babel_samples") / pl.col("samples")).alias("babel_pct"),
        (100 * pl.col("broken_samples") / pl.col("samples")).alias("broken_pct"),
    )
    .sort("motion_hours", descending=True)
)

print(dataset_summary.with_columns(
    pl.col("motion_hours").round(2),
    pl.col("clean_motion_hours").round(2),
    pl.col("babel_pct").round(1),
    pl.col("broken_pct").round(1),
))

In [ ]:
top_hours = dataset_summary.head(30).sort("motion_hours")

fig, ax = plt.subplots(1, 2, figsize=(14, 9))
y = np.arange(len(top_hours))

ax[0].barh(y, top_hours["motion_hours"], color="#999999", label="all")
ax[0].barh(y, top_hours["clean_motion_hours"], color="#4878a8", label="clean")
ax[0].set_yticks(y, top_hours["source_subset"])
ax[0].set_xlabel("motion hours")
ax[0].set_title("Motion hours by AMASS subset")
ax[0].legend()

ax[1].barh(y, top_hours["samples"], color="#999999", label="all")
ax[1].barh(y, top_hours["clean_samples"], color="#4878a8", label="clean")
ax[1].set_yticks(y, top_hours["source_subset"])
ax[1].set_xlabel("samples")
ax[1].set_title("Sample counts by AMASS subset")
ax[1].legend()

fig.tight_layout()

### BABEL coverage by AMASS subset

In [ ]:
coverage = (
    catalog
    .group_by("source_subset")
    .agg(
        pl.len().alias("n"),
        pl.col("has_babel_split").sum().alias("babel"),
        pl.col("broken").sum().alias("broken"),
        (pl.col("duration").sum() / 3600).alias("hours"),
    )
    .with_columns(
        (100 * pl.col("babel") / pl.col("n")).alias("babel_pct"),
        (100 * pl.col("broken") / pl.col("n")).alias("broken_pct"),
    )
    .sort("n", descending=True)
)

top = coverage.head(30).sort("babel_pct")

fig, ax = plt.subplots(figsize=(8, 9))
y = np.arange(len(top))

ax.barh(y, top["babel_pct"], color="#6acc64")
ax.set_yticks(y, top["source_subset"])
ax.set_xlabel("% samples with BABEL split")
ax.set_title("BABEL coverage by AMASS subset")
ax.set_xlim(0, 100)

fig.tight_layout()

### Labelled vs unlabelled duration

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

for label, filt, color in [
    ("BABEL split", pl.col("has_babel_split"), "#6acc64"),
    ("no BABEL split", ~pl.col("has_babel_split"), "#999999"),
]:
    x = catalog.filter(filt)["duration"].to_numpy()
    ax[0].hist(x, bins=60, alpha=0.6, label=f"{label} ({len(x):,})", color=color)
    ax[1].ecdf(x, label=label, color=color)

ax[0].set(xlabel="duration (s)", ylabel="samples", title="Duration distribution")
ax[1].set(xlabel="duration (s)", ylabel="fraction <= x", title="Duration CDF", xlim=(0, 35))
ax[0].legend()
ax[1].legend()

fig.tight_layout()

### Torque quality

`measure_quality` runs on the raw OpenSim channels before `build_features`, so `tau_rate` and `tau_absmax` below stay in physical units, unaffected by the signed-log transform. That is deliberate: discontinuity detection belongs in real torque units, not compressed ones.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

clean_catalog = catalog.filter(~pl.col("broken"))
broken_catalog = catalog.filter(pl.col("broken"))

ax[0].hist(np.log10(clean_catalog["tau_rate"].to_numpy().clip(1)), bins=60, alpha=0.8, color="#4878a8", label="clean")
ax[0].hist(np.log10(broken_catalog["tau_rate"].to_numpy().clip(1)), bins=60, alpha=0.8, color="#d1615d", label="broken")
ax[0].set(xlabel="log10 max |dtau/dt|", ylabel="samples", title="Torque discontinuity")
ax[0].legend()

bad_by_subset = (
    catalog
    .group_by("source_subset")
    .agg(pl.len().alias("n"), pl.col("broken").sum().alias("bad"))
    .with_columns((100 * pl.col("bad") / pl.col("n")).alias("bad_pct"))
    .sort("bad_pct", descending=True)
    .head(20)
    .sort("bad_pct")
)

ax[1].barh(bad_by_subset["source_subset"], bad_by_subset["bad_pct"], color="#d1615d")
ax[1].set(xlabel="% broken", title="Highest broken-rate subsets")

fig.tight_layout()

### What each training view sees

In [ ]:
def split_ids(split_set: str, split: str) -> list[str]:
    return (
        splits
        .filter((pl.col("split_set") == split_set) & (pl.col("split") == split))
        .select("sample_id")
        .unique()
        .get_column("sample_id")
        .to_list()
    )

views = {
    "pretrain_v1": split_ids("pretrain_v1", "train"),
    "babel train": split_ids("babel_official", "train"),
    "babel val": split_ids("babel_official", "val"),
    "babel test": split_ids("babel_official", "test"),
}

rows = []
for name, ids in views.items():
    view = catalog.filter(pl.col("sample_id").is_in(ids))
    clean_view = view.filter(~pl.col("broken"))
    rows.append({
        "view": name,
        "samples": len(view),
        "clean_samples": len(clean_view),
        "hours": view["duration"].sum() / 3600,
        "clean_hours": clean_view["duration"].sum() / 3600,
    })

view_df = pl.DataFrame(rows)
print(view_df)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(view_df))

ax[0].bar(x - 0.2, view_df["samples"], width=0.4, label="all", color="#999999")
ax[0].bar(x + 0.2, view_df["clean_samples"], width=0.4, label="clean", color="#4878a8")
ax[0].set_xticks(x, view_df["view"], rotation=25, ha="right")
ax[0].set(title="View sample counts", ylabel="samples")
ax[0].legend()

ax[1].bar(x - 0.2, view_df["hours"], width=0.4, label="all", color="#999999")
ax[1].bar(x + 0.2, view_df["clean_hours"], width=0.4, label="clean", color="#4878a8")
ax[1].set_xticks(x, view_df["view"], rotation=25, ha="right")
ax[1].set(title="View motion hours", ylabel="hours")
ax[1].legend()

fig.tight_layout()

### Normalization stats


In [ ]:
stats = t.load(NORMALIZATION_PATH, weights_only=True)
mean = stats["mean"].squeeze(0).numpy()
std = stats["std"].squeeze(0).numpy()
mask = stats.get("normalization_mask", t.ones_like(stats["mean"], dtype=t.bool)).squeeze(0).numpy()

print(f"mean shape: {mean.shape}")
print(f"std shape:  {std.shape}")
print("normalized channels:", [name for name, active in zip(["sin", "cos", "vel", "acc", "tau"], mask.any(axis=0)) if active])
print("pass-through channels:", [name for name, active in zip(["sin", "cos", "vel", "acc", "tau"], mask.any(axis=0)) if not active])

feature_names = ["sin", "cos", "vel", "acc", "tau"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

for channel, name in enumerate(feature_names):
    alpha = 0.75 if mask[:, channel].any() else 0.25
    ax[0].hist(mean[:, channel], bins=40, alpha=alpha, label=name)
    ax[1].hist(std[:, channel], bins=40, alpha=alpha, label=name)

ax[0].set(title="Feature means", xlabel="mean across training frames", ylabel="dofs")
ax[1].set(title="Feature stds", xlabel="std across training frames", ylabel="dofs")
ax[0].legend()
ax[1].legend()

fig.tight_layout()


### Normalization effect on feature distributions


In [ ]:
stats = t.load(NORMALIZATION_PATH, weights_only=True)
mean = stats["mean"]
std = stats["std"]
normalization_mask = stats.get("normalization_mask", t.ones_like(mean, dtype=t.bool)).bool()

def apply_feature_normalization(features: t.Tensor) -> t.Tensor:
    return t.where(normalization_mask, (features - mean) / std, features)

preview_n = min(24, len(train_samples))
preview_samples = train_samples.sample(n=preview_n, seed=7) if preview_n else train_samples

stored_chunks = []
normalized_chunks = []
for row in preview_samples.iter_rows(named=True):
    payload = t.load(OUT / row["motion_path"], weights_only=True)
    features = payload["features"].float()
    normalized = apply_feature_normalization(features)

    step = max(1, features.shape[0] // 800)
    stored_chunks.append(features[::step])
    normalized_chunks.append(normalized[::step])

stored_preview = t.cat(stored_chunks, dim=0).numpy()
normalized_preview = t.cat(normalized_chunks, dim=0).numpy()

print(f"preview samples: {preview_n}")
print(f"preview frames: {stored_preview.shape[0]:,}")

feature_names = ["sin", "cos", "vel", "acc", "tau"]
fig, ax = plt.subplots(len(feature_names), 2, figsize=(12, 14))

for channel, name in enumerate(feature_names):
    stored_values = stored_preview[:, :, channel].reshape(-1)
    normalized_values = normalized_preview[:, :, channel].reshape(-1)

    stored_lo, stored_hi = np.percentile(stored_values, [0.5, 99.5])
    norm_lo, norm_hi = np.percentile(normalized_values, [0.5, 99.5])

    ax[channel, 0].hist(stored_values, bins=80, range=(stored_lo, stored_hi), color="#4878a8", alpha=0.85)
    ax[channel, 1].hist(normalized_values, bins=80, range=(norm_lo, norm_hi), color="#d1615d", alpha=0.85)

    ax[channel, 0].set_ylabel(name)
    ax[channel, 0].set_title("stored (signed-log)" if channel == 0 else "")
    ax[channel, 1].set_title("after masked normalization" if channel == 0 else "")
    ax[channel, 1].axvline(0, color="black", linewidth=1, alpha=0.4)

ax[-1, 0].set_xlabel("feature value, clipped to 0.5-99.5 percentile")
ax[-1, 1].set_xlabel("feature value after dataset transform")
fig.suptitle("Stored (signed-log) vs dataset-normalized feature distributions", y=1.01)
fig.tight_layout()


### Normalization summary on preview samples


In [ ]:
rows = []
for channel, name in enumerate(feature_names):
    stored_values = stored_preview[:, :, channel].reshape(-1)
    normalized_values = normalized_preview[:, :, channel].reshape(-1)
    rows.append({
        "feature": name,
        "stored_mean": stored_values.mean(),
        "stored_std": stored_values.std(),
        "after_mean": normalized_values.mean(),
        "after_std": normalized_values.std(),
        "after_p01": np.percentile(normalized_values, 1),
        "after_p99": np.percentile(normalized_values, 99),
    })

normalization_effect = pl.DataFrame(rows)
print(normalization_effect.with_columns(pl.all().exclude("feature").round(3)))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(feature_names))

ax[0].bar(x - 0.2, normalization_effect["stored_mean"], width=0.4, label="stored", color="#4878a8")
ax[0].bar(x + 0.2, normalization_effect["after_mean"], width=0.4, label="after", color="#d1615d")
ax[0].axhline(0, color="black", linewidth=1, alpha=0.4)
ax[0].set_xticks(x, feature_names)
ax[0].set_title("Mean by feature channel")
ax[0].legend()

ax[1].bar(x - 0.2, normalization_effect["stored_std"], width=0.4, label="stored", color="#4878a8")
ax[1].bar(x + 0.2, normalization_effect["after_std"], width=0.4, label="after", color="#d1615d")
ax[1].axhline(1, color="black", linewidth=1, alpha=0.4)
ax[1].set_xticks(x, feature_names)
ax[1].set_title("Std by feature channel")
ax[1].legend()

fig.tight_layout()


### Normalization effect on one motion


In [ ]:
example = train_samples.sort("duration", descending=True).row(0, named=True)
payload = t.load(OUT / example["motion_path"], weights_only=True)
features = payload["features"].float()
normalized = apply_feature_normalization(features)
time = payload["time"].numpy()
dof_names = feature_dofs(human_def)

fig, ax = plt.subplots(len(feature_names), 2, figsize=(14, 12), sharex=True)

for channel, name in enumerate(feature_names):
    dof = int(features[:, :, channel].std(dim=0).argmax())
    stored_series = features[:, dof, channel].numpy()
    normalized_series = normalized[:, dof, channel].numpy()

    ax[channel, 0].plot(time, stored_series, color="#4878a8", linewidth=1)
    ax[channel, 1].plot(time, normalized_series, color="#d1615d", linewidth=1)
    ax[channel, 1].axhline(0, color="black", linewidth=1, alpha=0.35)

    ax[channel, 0].set_ylabel(f"{name}\n{dof_names[dof]}")
    ax[channel, 0].set_title("stored (signed-log)" if channel == 0 else "")
    ax[channel, 1].set_title("after masked normalization" if channel == 0 else "")

ax[-1, 0].set_xlabel("time (s)")
ax[-1, 1].set_xlabel("time (s)")
fig.suptitle(f"{example['source_subset']} / {example['source_path']}", y=1.01)
fig.tight_layout()
